<a href="https://colab.research.google.com/github/mAliAytekin/ai-research-notes/blob/main/Neural_Architecture_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- NAS aims to automate the design and fine tuning of neural networks
- uses search algorithms to explore and discover optimal neural network architecture
- NAS components
  - search space => the set of possible neural network architectures (NNA)
    - layer-wise
    - cell-based
    - hierarchical structures etc..
  - search strategy => method for finding optimal NNA
    - reinforcement learning
    - random search
    - bayesian
    - evolutionary algorithms
  - evaluation strategy => assessing the performance of candidate NNA


### GA based NAS

In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import random
from tqdm import tqdm

In [39]:
def get_iris_data(test_size=0.2):
  iris = load_iris()
  X,y = iris.data, iris.target

  X_train, X_test, y_train, y_test  = train_test_split(X,y,test_size=test_size,random_state=42,stratify=y)

  scaler = StandardScaler()
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)

  train_dataset = TensorDataset(torch.tensor(X_train,dtype=torch.float32),torch.tensor(y_train,dtype=torch.long))
  test_dataset = TensorDataset(torch.tensor(X_test,dtype=torch.float32),torch.tensor(y_test,dtype=torch.long))

  return train_dataset, test_dataset

In [40]:
def create_model_from_chromosome(chromosome,input_size=4,output_size=3):
    n1_idx, a1_idx, n2_idx, a2_idx = chromosome

    n1_neurons = NEURON_OPTIONS[n1_idx]
    n2_neurons = NEURON_OPTIONS[n2_idx]
    act1 = ACTIVATION_MAP[a1_idx]
    act2 = ACTIVATION_MAP[a2_idx]

    model = nn.Sequential(
        nn.Linear(input_size, n1_neurons),
        act1,
        nn.Linear(n1_neurons, n2_neurons),
        act2,
        nn.Linear(n2_neurons, output_size)
    )
    return model

def evaluate_fitness(chromosome,train_loader,test_loader,epochs=15):
  model = create_model_from_chromosome(chromosome)
  optimizer = optim.Adam(model.parameters(),lr=0.001)
  criterion = nn.CrossEntropyLoss()

  # train model
  for _ in range(epochs):
    for data,target in train_loader:
      optimizer.zero_grad()
      output = model(data)
      loss = criterion(output,target)
      loss.backward()
      optimizer.step()

  # test model
  model.eval()
  with torch.no_grad():
    for data,target in test_loader:
      output = model(data)
      _, predicted = torch.max(output,1)
      correct = (predicted==target).sum().item()
      accuracy = correct/len(target)

  return accuracy

In [41]:
class GeneticAlgorithmNAS:

  def __init__(self,population_size,mutation_rate, crossover_rate, elitism_count):
    self.population_size = population_size
    self.mutation_rate = mutation_rate
    self.crossover_rate = crossover_rate
    self.elitism_count = elitism_count
    self.population = self._init_population()

  def _create_chromosome(self):
      # [n1_idx, a1_idx, n2_idx, a2_idx]
      return [
          random.randint(0, len(NEURON_OPTIONS) - 1),
          random.randint(0, len(ACTIVATION_MAP) - 1),
          random.randint(0, len(NEURON_OPTIONS) - 1),
          random.randint(0, len(ACTIVATION_MAP) - 1),
      ]

  def _init_population(self):
    return [self._create_chromosome() for _ in range(self.population_size)]

  def _crossover(self,parent1,parent2):
    if random.random()<self.crossover_rate:
      point = random.randint(1,len(parent1)-1)
      child1 = parent1[:point] + parent2[point:]
      child2 = parent2[:point] + parent1[point:]
      return child1,child2
    return parent1,parent2

  def _mutate(self, chromosome):
      for i in range(len(chromosome)):
          if random.random() < self.mutation_rate:
              if i % 2 == 0:
                  chromosome[i] = random.randint(0, len(NEURON_OPTIONS) - 1)
              else:
                  chromosome[i] = random.randint(0, len(ACTIVATION_MAP) - 1)
      return chromosome

  def _selection(self, population_with_fitness):
      tournament_size = 3
      selected = random.sample(population_with_fitness, tournament_size)
      selected.sort(key=lambda x: x[1], reverse=True) # order by fitness
      return selected[0][0] # best one

  def run(self, generations):
      print("--- starting GA-NAS ---")
      best_overall_chromosome = None
      best_overall_fitness = -1

      for gen in range(generations):
          print(f"\n--- generation {gen+1}/{generations} ---")
          pbar = tqdm(self.population, desc="calculating fitness")
          population_with_fitness = [(chrom, evaluate_fitness(chrom, train_loader, test_loader)) for chrom in pbar]

          population_with_fitness.sort(key=lambda x: x[1], reverse=True)

          current_best_chromosome = population_with_fitness[0][0]
          current_best_fitness = population_with_fitness[0][1]

          if current_best_fitness > best_overall_fitness:
              best_overall_fitness = current_best_fitness
              best_overall_chromosome = current_best_chromosome

          print(f"gen {gen+1} best fitness: {current_best_fitness:.4f}")

          next_generation = []

          elites = [chrom for chrom, fit in population_with_fitness[:self.elitism_count]]
          next_generation.extend(elites)

          while len(next_generation) < self.population_size:
              parent1 = self._selection(population_with_fitness)
              parent2 = self._selection(population_with_fitness)
              child1, child2 = self._crossover(parent1, parent2)
              next_generation.append(self._mutate(child1))
              if len(next_generation) < self.population_size:
                  next_generation.append(self._mutate(child2))

          self.population = next_generation

      print("\n--- ends ---")
      return best_overall_chromosome, best_overall_fitness


In [42]:
BATCH_SIZE=16
train_dataset, test_dataset = get_iris_data()
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [43]:
# search space
NEURON_OPTIONS = [4, 8,12, 32, 64]
ACTIVATION_MAP = {0: nn.ReLU(), 1: nn.Tanh(), 2: nn.Sigmoid()}

ga_nas = GeneticAlgorithmNAS(
  population_size=10,
  mutation_rate=0.1,
  crossover_rate=0.8,
  elitism_count=2,
)

best_chromosome, best_fitness = ga_nas.run(generations=5)

n1_idx, a1_idx, n2_idx, a2_idx = best_chromosome
act_names = {0: 'ReLU', 1: 'Tanh', 2: 'Sigmoid'}

final_architecture_desc = {
  "layer 1 neuron ": NEURON_OPTIONS[n1_idx],
  "layer 1 activation": act_names[a1_idx],
  "layer 2 neuron": NEURON_OPTIONS[n2_idx],
  "layer 2 activation": act_names[a2_idx],
}

print("\nbest architecture:")
print(final_architecture_desc)
print(f"fitness value : {best_fitness:.4f}")

--- starting GA-NAS ---

--- generation 1/5 ---


calculating fitness: 100%|██████████| 10/10 [00:01<00:00,  5.60it/s]


gen 1 best fitness: 0.8571

--- generation 2/5 ---


calculating fitness: 100%|██████████| 10/10 [00:01<00:00,  5.64it/s]


gen 2 best fitness: 0.9286

--- generation 3/5 ---


...

gen 3 best fitness: 0.9286

--- generation 4/5 ---


calculating fitness: 100%|██████████| 10/10 [00:01<00:00,  5.34it/s]


gen 4 best fitness: 0.9286

--- generation 5/5 ---


calculating fitness: 100%|██████████| 10/10 [00:01<00:00,  5.34it/s]

gen 5 best fitness: 0.9286

--- ends ---

best architecture:
{'layer 1 neuron ': 64, 'layer 1 activation': 'ReLU', 'layer 2 neuron': 64, 'layer 2 activation': 'ReLU'}
fitness value : 0.9286


### automl systems

- https://automl.github.io/auto-sklearn/master/

- https://github.com/automl/Auto-PyTorch


### resources

- https://www.geeksforgeeks.org/machine-learning/what-is-automl-in-machine-learning/

- https://medium.com/thedeephub/efficient-neural-architecture-search-enas-overview-7ce747b75b27

- coding with Gemini 2.5 Pro